# Challenge Task — Section 2: Classical Classifier Swap

We replace the baseline's per-class `DecisionTreeClassifier` with
`HistGradientBoostingClassifier` (sklearn-native, comparable to XGBoost on
these aggregated features; XGBoost itself is not installed). We:

1. Train the baseline (DT) replication for a direct comparison.
2. Train HistGB at a default config to confirm a lift.
3. **Hyperparameter analysis** over `max_iter` × `max_leaf_nodes`.
4. **Per-class threshold tuning** on the local validation set.
5. Compare DT vs HistGB (default) vs HistGB (tuned) — overall + per class.

All evaluation uses the official segment-based macro-F1 from
`baseline/evaluate.py`. The seed=42 local split (499 val + 500 non-hidden
test) is reproduced exactly from the baseline notebook.

## 1 — Setup

In [ ]:
import os, sys, time, glob, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier

PHASE = Path('/Users/amrabdullah/AI_2026ss/mlpc/phases/04_challenge')
sys.path.insert(0, str(PHASE))
import utils

ART = PHASE / 'artifacts'; ART.mkdir(exist_ok=True)
print('utils OK, CLASS_NAMES:', len(utils.CLASS_NAMES))

## 2 — Data

In [ ]:
train_files = sorted(glob.glob(str(utils.PATH_TRAIN / 'audio_features' / '*.npz')))
our_val_files, our_nht_files = utils.split_validation(seed=42)
hidden_files = sorted(glob.glob(str(utils.PATH_TEST / 'audio_features' / '*.npz')))
print(f'train={len(train_files)}  val={len(our_val_files)}  non-hidden={len(our_nht_files)}  hidden={len(hidden_files)}')

# Match the baseline's training matrix: load all, subsample 50k segments
rng = np.random.default_rng(seed=42)
print('Loading all training segments ...')
t0 = time.time()
X_train, Y_train = utils.load_all_segments(train_files)
print(f'  X_train={X_train.shape}  Y_train={Y_train.shape}  ({time.time()-t0:.0f}s)')

MAX_TRAIN = 50_000
if X_train.shape[0] > MAX_TRAIN:
    idx = rng.choice(X_train.shape[0], size=MAX_TRAIN, replace=False)
    X_train, Y_train = X_train[idx], Y_train[idx]
    print(f'  subsampled to {X_train.shape[0]} segments')

ann_df_val = pd.read_csv(utils.PATH_VAL / 'annotations.csv')
print(f'val annotations: {len(ann_df_val)} rows / {ann_df_val.filename.nunique()} files')

## 3 — Baseline replica (DecisionTree)
Replicates `baseline/challenge_baseline.ipynb` for a direct comparison.

In [ ]:
print('Training DT baseline (max_depth=20, max_features=sqrt)...')
t0 = time.time()
dt = MultiOutputClassifier(DecisionTreeClassifier(max_depth=20, max_features='sqrt', random_state=42), n_jobs=-1)
dt.fit(X_train, Y_train)
print(f'  fit: {time.time()-t0:.1f}s')

pred_dt_val = utils.generate_predictions(our_val_files, dt, use_probs=False)
f1_dt_val, per_dt_val = utils.evaluate_split(pred_dt_val, our_val_files, ann_df_val)
print(f'DT  local-val  macro-F1 = {f1_dt_val:.4f}')

## 4 — HistGB default config
Default HistGB: `max_iter=200, learning_rate=0.1, max_leaf_nodes=31`.

In [ ]:
def make_histgb(max_iter=200, max_leaf_nodes=31, learning_rate=0.1):
    return MultiOutputClassifier(
        HistGradientBoostingClassifier(
            max_iter=max_iter,
            learning_rate=learning_rate,
            max_leaf_nodes=max_leaf_nodes,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=42,
        ),
        n_jobs=-1,
    )

print('Training HistGB (default) ...')
t0 = time.time()
hgb_def = make_histgb()
hgb_def.fit(X_train, Y_train)
print(f'  fit: {time.time()-t0:.1f}s')

pred_hgb_val = utils.generate_predictions(our_val_files, hgb_def, use_probs=False)
f1_hgb_def_val, per_hgb_def_val = utils.evaluate_split(pred_hgb_val, our_val_files, ann_df_val)
print(f'HistGB default  local-val  macro-F1 = {f1_hgb_def_val:.4f}  (DT was {f1_dt_val:.4f})')

## 5 — Hyperparameter analysis
We vary two hyperparameters that matter most for gradient-boosted trees on
this kind of tabular feature space:

- `max_iter` ∈ {100, 200, 300, 500} — number of boosting rounds
- `max_leaf_nodes` ∈ {15, 31, 63} — leaf budget per tree (capacity per round)

Each config is trained, evaluated on the local validation set, and macro-F1
is recorded. We then pick the best config to take forward to threshold
tuning.

In [ ]:
GRID = {
    'max_iter':       [100, 200, 300, 500],
    'max_leaf_nodes': [15, 31, 63],
}
hp_results = []
for mi in GRID['max_iter']:
    for ml in GRID['max_leaf_nodes']:
        t0 = time.time()
        m  = make_histgb(max_iter=mi, max_leaf_nodes=ml)
        m.fit(X_train, Y_train)
        fit_t = time.time() - t0
        pred = utils.generate_predictions(our_val_files, m, use_probs=False)
        f1, _ = utils.evaluate_split(pred, our_val_files, ann_df_val)
        hp_results.append({'max_iter': mi, 'max_leaf_nodes': ml,
                           'val_macro_f1': f1, 'fit_seconds': round(fit_t, 1)})
        print(f'  max_iter={mi:4d}  max_leaf_nodes={ml:3d}  -> val macro-F1 = {f1:.4f}  ({fit_t:.0f}s)')

hp_df = pd.DataFrame(hp_results)
hp_df.to_csv(ART / 'section2_hp_grid.csv', index=False)
print('saved hp grid to artifacts/section2_hp_grid.csv')
hp_df

In [ ]:
# Heat-map plot for the report
piv = hp_df.pivot(index='max_iter', columns='max_leaf_nodes', values='val_macro_f1')
fig, ax = plt.subplots(figsize=(6.5, 4))
im = ax.imshow(piv.values, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
ax.set_yticks(range(len(piv.index)));   ax.set_yticklabels(piv.index)
ax.set_xlabel('max_leaf_nodes'); ax.set_ylabel('max_iter')
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        ax.text(j, i, f'{piv.values[i,j]:.3f}', ha='center', va='center',
                color='white' if piv.values[i,j] < piv.values.mean() else 'black')
ax.set_title('HistGB hyperparameter sweep — local-val macro-F1 (no threshold tuning)')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
fig.savefig(PHASE / 'report' / 'figures' / 'hp_grid_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Per-class threshold tuning
HistGB exposes calibrated probabilities; with the default 0.5 cutoff,
rare classes (whose posteriors are far below 0.5) get systematically
predicted as absent. Sweeping τ ∈ [0.05, 0.95] per class on the local
validation set, picking the F1-optimal one per class, is the single
largest decision-rule lever (+0.20 in previous phase).

In [ ]:
# Pick the best config from the grid
best_row = hp_df.sort_values('val_macro_f1', ascending=False).iloc[0]
best_cfg = {'max_iter': int(best_row['max_iter']),
            'max_leaf_nodes': int(best_row['max_leaf_nodes'])}
print('Best grid config:', best_cfg, ' val macro-F1 (no thr):', round(best_row['val_macro_f1'], 4))

print('Refitting best config and computing probabilities for tuning ...')
hgb_best = make_histgb(**best_cfg)
hgb_best.fit(X_train, Y_train)

# Build the validation probability matrix from the WHOLE-second segments
def collect_probs(file_list, classifier):
    probs_all, labels_all = [], []
    for fp in file_list:
        data = dict(np.load(fp, allow_pickle=True))
        ts = data['start_time']
        mask = np.isclose(ts % 1.0, 0.0)
        X = utils.build_feature_matrix(data)[mask]
        Y = utils.get_segment_labels(data)[mask]
        P = utils.predict_proba_multiout(classifier, X)
        probs_all.append(P); labels_all.append(Y)
    return np.vstack(probs_all), np.vstack(labels_all)

P_val, Y_val = collect_probs(our_val_files, hgb_best)
print(f'val probability matrix: {P_val.shape}')

thr = utils.tune_per_class_thresholds(P_val, Y_val,
                                      grid=np.arange(0.05, 0.96, 0.02))
print('Per-class thresholds:')
for c, t in zip(utils.CLASS_NAMES, thr):
    print(f'  {c:<30}  tau = {t:.2f}')
np.save(ART / 'section2_thresholds.npy', thr)

In [ ]:
# Apply tuned thresholds and re-evaluate on local val
pred_tuned_val = utils.generate_predictions(our_val_files, hgb_best,
                                             thresholds=thr, use_probs=True)
f1_tuned_val, per_tuned_val = utils.evaluate_split(pred_tuned_val, our_val_files, ann_df_val)
print(f'HistGB best+tuned   local-val  macro-F1 = {f1_tuned_val:.4f}  '
      f'(default thresholds gave {best_row["val_macro_f1"]:.4f})')

## 7 — Final comparison on the non-hidden test set
Untouched until now, so the number is an honest single-shot estimate.

In [ ]:
# DT baseline on non-hidden test
pred_dt_nht = utils.generate_predictions(our_nht_files, dt, use_probs=False)
f1_dt_nht, per_dt_nht = utils.evaluate_split(pred_dt_nht, our_nht_files, ann_df_val)

# HistGB (best + tuned thresholds) on non-hidden test
pred_hgb_nht = utils.generate_predictions(our_nht_files, hgb_best,
                                          thresholds=thr, use_probs=True)
f1_hgb_nht, per_hgb_nht = utils.evaluate_split(pred_hgb_nht, our_nht_files, ann_df_val)

print(f'\n=== Non-hidden test ===')
print(f'  DT baseline             macro-F1 = {f1_dt_nht:.4f}')
print(f'  HistGB best + thresholds macro-F1 = {f1_hgb_nht:.4f}  '
      f'(+{f1_hgb_nht - f1_dt_nht:.4f})')

cmp_nht = per_dt_nht[['annotation', 'f1']].rename(columns={'f1': 'DT_f1'}).merge(
          per_hgb_nht[['annotation', 'f1']].rename(columns={'f1': 'HistGB_f1'}),
          on='annotation')
cmp_nht['delta'] = cmp_nht['HistGB_f1'] - cmp_nht['DT_f1']
print('\nPer-class delta (HistGB - DT) on non-hidden test:')
print(cmp_nht.sort_values('delta', ascending=False).to_string(index=False))
cmp_nht.to_csv(ART / 'section2_per_class_nht.csv', index=False)

In [ ]:
# Plot per-class F1 for the report
order = utils.CLASS_NAMES
dt_f1  = cmp_nht.set_index('annotation').loc[order, 'DT_f1'].values
hgb_f1 = cmp_nht.set_index('annotation').loc[order, 'HistGB_f1'].values

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(order)); w = 0.4
ax.bar(x - w/2, dt_f1,  w, label=f'DT  (macro {f1_dt_nht:.3f})',  color='coral')
ax.bar(x + w/2, hgb_f1, w, label=f'HistGB+thr  (macro {f1_hgb_nht:.3f})', color='steelblue')
ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha='right')
ax.set_ylabel('F1'); ax.set_ylim(0, 1.05)
ax.set_title('Per-class F1 on non-hidden test — DecisionTree vs HistGB (per-class thresholds)')
ax.legend(); plt.tight_layout()
fig.savefig(PHASE / 'report' / 'figures' / 'section2_perclass.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 — Save the trained best model + thresholds for downstream notebooks

In [ ]:
import joblib
joblib.dump(hgb_best, ART / 'section2_histgb_best.joblib')
joblib.dump(dt,       ART / 'section2_dt_baseline.joblib')
np.save(ART / 'section2_thresholds.npy', thr)

summary = {
    'best_config': best_cfg,
    'baseline_DT_val_macro_f1':            float(f1_dt_val),
    'HistGB_default_val_macro_f1':         float(f1_hgb_def_val),
    'HistGB_best_val_macro_f1_no_thresh':  float(best_row['val_macro_f1']),
    'HistGB_best_val_macro_f1_tuned':      float(f1_tuned_val),
    'baseline_DT_nht_macro_f1':            float(f1_dt_nht),
    'HistGB_best_nht_macro_f1_tuned':      float(f1_hgb_nht),
    'thresholds': thr.tolist(),
    'class_names': list(utils.CLASS_NAMES),
}
with open(ART / 'section2_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('saved -> artifacts/section2_{histgb_best.joblib, thresholds.npy, summary.json}')
print(json.dumps(summary, indent=2)[:800])